# KG catch-up + intent chat session -> turns + knowledge graph

**Who is it talking to?** The agent does not know that up front: the chat window opens by naming
everyone the knowledge graph has had a conversation with and asking for the human's name. With the
name it queries the graph for what it remembers of that person -- the last conversation, what was
talked about then, and the catch-up topics -- shows their timeline in the window, and starts the
catch-up described below (`cltl.kg_chat.name_first_session.NameFirstCatchUpSession`).

A new process flow on top of **[`kg_intent_chat.ipynb`](kg_intent_chat.ipynb)**: instead of
starting cold, the agent first figures out **how long it's been since it last spoke with this
human, what they'd talked about back then, and how MUCH of that it should expect to hear about
again** -- then keeps the conversation going, topic by topic, until it judges it has *enough* for
the catch-up period, not just "asked once about everything."

Built on **[`gaps_from_kg/get_temporal_containers.py`](../src/cltl/gaps_from_kg/get_temporal_containers.py)**
-- a different query layer over the same GraphDB repository (via `cltl.brain.LongTermMemory`)
than `kg_gap_finder.py`/`intent_gap_finder.py`'s rdflib/SPARQL-endpoint queries, which is what the
rest of the per-turn flow below still uses unchanged. See
**[`catch_up_from_kg.py`](catch_up_from_kg.py)** for the full implementation; in short, a loop:

1. **Ask about a catch-up-period topic** (`SaturationTracker.next_question()`) -- one of
   `chat_sessions.DEFAULT_GAP_ACTIVITY_TYPES` (exercise, diet, sleep, symptoms, medication, ...)
   this human has real history with, but not yet ENOUGH reported for this period (see
   "Saturation" below).
2. **Whatever they report is handled entirely by `KgIntentChatSession`'s own EXISTING per-turn
   flow, unchanged**: SRL extraction -> push to the KG -> `intent_gap_finder.next_intent_gap()`
   keeps asking its own follow-up questions about *that* activity (what/how much/when/where) for
   as long as its matching intent still has unmet requirements.
3. **Once that activity's own follow-ups are exhausted**, go back to step 1 -- another
   under-covered topic, or the same one again if it's still short -- **unless every topic has
   reached saturation**, at which point (see "Wrapping up" below) the LLM decides how to close
   things out, and the conversation continues normally after that.

**The catch-up period is capped at `catch_up.MAX_SATURATION_GAP_DAYS` (14 days)**, even when the
real gap since the last conversation is much longer -- a human who hasn't talked in two months
still only needs the last two weeks caught up on live, not the whole two months.

**Saturation** -- "enough knowledge for the catch-up period" -- is defined *per topic* against a
fixed **weekly** baseline (`catch_up.CATCH_UP_WINDOW_DAYS`, 7 days), not one tied to however long
the catch-up period itself is: `_windowed_average_rate()` tiles that week-long window backwards
across this human's *entire* history to get their typical weekly rate, which is then scaled to
however many days the (capped) catch-up period covers. If they've historically reported exercise
~3 times a week and the catch-up period is 2 weeks, 6 reported live is enough for exercise
specifically -- not "ask about it once and move on" and not "keep asking forever." A hard
per-topic cap (`SaturationTracker.MAX_ASKS_PER_TOPIC`, default 3) still gives up on a topic the
human simply has nothing more to say about, regardless of its target.

**Wrapping up.** The FIRST time every topic is saturated (or capped out),
`SaturationTracker.wrap_up_message()` sends one message naming what was actually covered and
lets the LLM decide for itself: continue with a short question if there's an obvious thread left,
or wrap up warmly and say goodbye. After that one message, replies fall through to the plain
default `agent_fn` unchanged, so the conversation continues (or ends, if the human says goodbye
back) like an ordinary chat.

**Before running this:** an LLM (see "Which LLM" below: `OPENAI_API_KEY` set, or a local
llama-server running), and a reachable knowledge-graph SPARQL endpoint (e.g. a local GraphDB
`event_sandbox` repository) at `KG_ADDRESS` below, ideally one with some history for the people
the agent knows (for a new name there's nothing to catch up on, and this degrades to a plain,
short "what's new?" opener with an immediately-saturated tracker).

The next image shows the overview of the dialogue flow:

<img src="../doc/kg_catchup_decision_flow.png" width="900">

In [2]:
# In cltl-kg-driven-chat, chat_sessions.py and catch_up_from_kg.py sat next to these notebooks;
# in this project they are the package cltl.kg_chat in src/cltl/kg_chat. src/ comes first on
# sys.path, so that src/cltl/commons replaces the installed cltl.commons (see README.md).
# Run the notebooks from notebooks/: src/cltl and intents/ are found from there.
import sys
from pathlib import Path

_SRC = str(Path.cwd().parent / "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

## Which LLM

`LLM_BACKEND = "openai"` uses OpenAI's API (needs `OPENAI_API_KEY`); `"local"` uses a local server
with an OpenAI-compatible API at `LLM_BASE_URL`, e.g. llama.cpp's

    llama-server -hf unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF:Q8_0 --port 8080 -c 16384

Start it with `-c` of at least 8192: the event extraction prompt alone is ~6k tokens. This is the
notebook's own choice; `src/main.py` takes it from `llm_backend` in `config/default.config`.
Re-running this cell switches backends without restarting the kernel, but a chat session made
before keeps the extractor it was made with, so re-run the cells below it too.

In [3]:
from cltl.kg_chat import llm_backend

LLM_BACKEND = "local"                     # "openai" or "local"
LLM_BASE_URL = "http://localhost:8080/v1"  # llama-server's API (LLM_BACKEND = "local" only)
LLM_MODEL = None                           # None: gpt-5.1 for openai; llama-server ignores it
LLM_TIMEOUT = 120                          # seconds per call; a local model is slower

llm_backend.configure(LLM_BACKEND, base_url=LLM_BASE_URL if LLM_BACKEND == "local" else None,
                      model=LLM_MODEL, timeout=LLM_TIMEOUT)
print("LLM:", llm_backend.describe())

LLM: local at http://localhost:8080/v1 (serving unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF:Q8_0)


In [4]:
import time
from datetime import datetime, timedelta

from cltl.kg_chat.chat_sessions import save_turns
from cltl.kg_chat import catch_up_from_kg as catch_up
import matplotlib.pyplot as plt
from cltl.plots.patient_timeline import plot_patient_timeline_from_kg


KG_ADDRESS = "http://localhost:7200/repositories/event_sandbox"
KG_LOG_DIR = "kg_logs"

# Directory of intent *.json files -- None auto-discovers the project's own intents/ folder (see
# intent_gap_finder.load_intents()/_default_intents_dir()).
INTENTS_DIR = None

# A FRESH id every run, not a fixed constant -- see kg_intent_chat.ipynb's own CHAT_ID cell for
# why (reusing one across runs collides every run's activities on the same subject URIs).
CHAT_ID = int(time.time())

# "Now", for both the catch-up queries below and every activity pushed to the graph this run.
CURRENT_DATE = datetime.now()

# Used only if the KG has no earlier conversation with the human on record at all (e.g. a brand
# new human, or a fresh/empty graph) -- how far back to assume the last (nonexistent) conversation
# was, so there's still a sensible "it's been N days" opener instead of a crash.
FALLBACK_GAP_DAYS = 7

# How many days the timeline in the GUI looks back at first; None shows everything in the KG.
TIMELINE_DAYS = 60

## Step 1: connect to the knowledge graph

Connects to the same KG `KG_ADDRESS` points at (via `cltl.brain.LongTermMemory` this time, not
rdflib). `connect_brain()` also makes sure the graph has
**[`gaps_from_kg/n2mu_sem_roles.py`](../src/cltl/gaps_from_kg/n2mu_sem_roles.py)**'s small
`rdfs:subPropertyOf` mapping uploaded (a one-time, idempotent step -- see that module's own
docstring): without it, every real activity looks dateless/actorless/placeless to
`get_temporal_containers()`'s underlying query, since activities only ever carry this project's
own `n2mu:agent`/`n2mu:location`/`n2mu:time/...` predicates, never `sem:hasActor`/`hasPlace`/
`hasTime` directly. Past that one-time schema write, everything here only reads until the chat
starts.

In [5]:
brain = catch_up.connect_brain(KG_ADDRESS, log_dir=KG_LOG_DIR)
print("People the agent has talked with:", catch_up.find_known_humans(brain))

2026-11-05 15:28:24 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-11-05 15:28:24 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-11-05 15:28:24 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-11-05 15:28:24 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-11-05 15:28:24 -     INFO -                                   cltl.brain.TrustCalculator - Booted


People the agent has talked with: ['Jan', 'Mehmet', 'Piek']


### The timeline in the chat window: what was the case for the human, roughly

As soon as the human has said who they are, the chat window's graph panel shows a timeline of
everything in the KG about them over the last `TIMELINE_DAYS` days, drawn by
**[`plots/patient_timeline.py`](../src/cltl/plots/patient_timeline.py)** with a single SPARQL
query. An activity counts as the patient's when they are one of its actors (`sem:hasActor`), or
when they said the utterance it was extracted from. It is redrawn after every reply, with what
this chat added outlined; **Graph** switches to the graph of the current activity, **Timeline**
back.

**Period.** How far back the timeline looks is `TIMELINE_DAYS` in the settings above (`None`
shows everything in the KG). The GUI's timeline starts out with the same period; its
**Look back** box changes it during the chat, to a preset (7 days to 1 year, or
everything) or any number of days typed in.

**Lanes.** Every activity type has its own lane (row), e.g. *exercise*, *take drink*,
*mental condition*:

- **Blue** lanes are **activities**, things the patient does (exercise, take food, take drink, diet,
  sleep, take medicine, treatment, measurement, social, ...).
- **Orange** lanes are **conditions**, states the patient is in (symptom and every
  `*_condition` type).
- Activities without a specific type go in the *other* lane.
- Activities and conditions share one plot as long as there are at most 12 lanes, with the
  activities on top. With more lanes they get a plot each, on the same time axis.
- Entries of one lane that overlap in time are drawn above each other, up to three rows per lane.

**When it was the case** is only roughly known: it comes from the activity's `n2mu:time/*` role,
as extracted from what was said.

| Mark | Time role | Meaning |
|---|---|---|
| filled dot | `dateTime` | on that date ("yesterday", "on Monday") |
| hollow dot | none that is a date | no date was given ("for an hour", "how often"), so the dot is on the day it was **mentioned** |
| solid bar | `rangeTime` | since that date, until it was mentioned ("since last week") |
| hatched bar | `recurringTime`, or an `EventSeries` | a habit, since that date until it was mentioned ("every morning") |
| faint bar | `vagueTime` only | vaguely, since that date ("lately", "recently") |

A bar that starts before the window is cut off at the left edge. Where there is room, an entry is
labelled with the activity's own label, e.g. *drink beer*. Entries without a label of their own
show no text.

**Dashed lines** mark the **last conversation** (looked up once the name is known) and
**now**. What lies between them is the gap this session tries to catch up on, so
an empty stretch there is what the catch-up questions ask about.

**Reading it with care.** The plot shows what the KG holds, not what really happened. Extraction
errors show up as they are: the agent's own questions are sometimes extracted as the patient's
activities (e.g. *what you eat and drink*), and a misread date can put a bar far from where it
belongs.

## Step 2: open the chat, ask the name, catch up

The window opens with the people the agent knows and asks who it's talking to
(`NameFirstCatchUpSession`). The name in the answer -- a known one in any case, e.g. "it's jan",
or a new one, which an LLM call picks out -- starts the catch-up:

1. **Memory.** `session.memory(name)` queries the graph: the last conversation with them,
   `find_catch_up_topics()` (every topic with history, with a saturation target for the catch-up
   period, capped at `MAX_SATURATION_GAP_DAYS`), and `find_last_conversation_topics()` (what they
   themselves talked about in the last chat). The timeline appears in the graph panel.
2. **Opening.** `SaturationTracker.opening_question()`: how long it's been, a one-sentence summary
   of what was talked about *last time* (only those topics), and a question about the ONE topic
   that most needs catching up on.
3. **Catch-up.** A `KgIntentChatSession` whose default reply is wrapped by
   `wrap_agent_fn_with_saturation_loop()`: as long as `tracker.is_saturated()` is False, a reply
   that would otherwise be the generic LLM one asks about the next under-covered topic instead,
   and `on_new_subject=tracker.record_new_activity` keeps the counts up to date. Questions about
   the human's own record ("what did I drink?") are answered from the graph.

Close the window (Quit, or type "bye") to continue with the cells below.

In [6]:
from kg_chat_gui import run_gui
from cltl.kg_chat.name_first_session import NameFirstCatchUpSession

session = NameFirstCatchUpSession(KG_ADDRESS, brain, chat=CHAT_ID, intents_dir=INTENTS_DIR,
                                  log_dir=KG_LOG_DIR, current_date=CURRENT_DATE,
                                  fallback_gap_days=FALLBACK_GAP_DAYS)
turns = run_gui(session, timeline_days=TIMELINE_DAYS)

# What the chat found out, for the cells below.
if session.session is None:
    print("The chat ended before the human said who they are.")
else:
    HUMAN = session.human
    kg_session, tracker = session.session, session.tracker
    last_conversation_date, catch_up_topics = session.last_conversation_date, session.catch_up_topics
    print(f"Talked with {HUMAN}; the last conversation before this one was on {last_conversation_date:%Y-%m-%d}")
    print("Talked about last time:", ", ".join(session.last_topics) or "nothing on record")
    for topic in catch_up_topics:
        print(f"  - {topic['activity_type']}: {topic['weekly_rate']:.1f}/week historically -> "
              f"target {topic['expected_count']}, reported {tracker.reported[topic['activity_type']]}")

Our previous conservation was on Tuesday 2026-11-03 00:00:00
Today is Thursday 2026-11-05 15:28:23.017916
What happened in the last 2 days?
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:exercise.                 ?id rdfs:label ?label .                 }
I found 54 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_food.                 ?id rdfs:label ?label .                 }
I found 57 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_drink.                 ?id rdfs:label ?label .                 }
I found 57 activities
PREFIX n2mu: <http://cltl.nl/leolani/n

2026-11-05 15:28:33 -     INFO -                              cltl.kg_chat.name_first_session - Catching up with Jan: last conversation 2026-11-03 00:00:00, last topics ['take_food']


PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:sleep.                 ?id rdfs:label ?label .                 }
I found 14 activities


2026-11-05 15:28:34 -     INFO -                                                       httpx2 - HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"


turn {'chat': 1793888903, 'human': 'Jan', 'date': '2026,Nov,05', 'turn': 1, 'speaker': 'agent', 'utterance': 'Hi Jan! 2 days ago we talked about food. Since then, how have your drinks been going?'}


2026-11-05 15:28:34 -     INFO -                                                       httpx2 - HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"


... (49 outputs left out)

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.14it/s]
2026-11-05 15:30:42 -     INFO -                                                       httpx2 - HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-05 15:30:42 -     INFO -                                                       httpx2 - HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-05 15:30:43 -     INFO -                                                       httpx2 - HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"


[turn 12] Jan: Around my twon
    pushed 1 triple(s):
      chat1793888903.1  location  =  around my town
    selected gap: subject=http://cltl.nl/leolani/n2mu/chat1793888903.1 predicate=http://cltl.nl/leolani/n2mu/location kind=predicate_object_type
turn {'chat': 1793888903, 'human': 'Jan', 'date': '2026,Nov,05', 'turn': 13, 'speaker': 'agent', 'utterance': 'Nice—cycling around your town sounds like a great way to stay active. So far, we’ve talked about how you’ve been feeling physically, what you’ve been eating and drinking, your mood, how you’ve been checking your blood sugar, and a bit about your daily routine and what’s been going on in your life lately.'}


2026-11-05 15:30:43 -     INFO -                                                       httpx2 - HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"


[turn 13] agent: Nice—cycling around your town sounds like a great way to stay active. So far, we’ve talked about how you’ve been feeling physically, what you’ve been eating and drinking, your mood, how you’ve been checking your blood sugar, and a bit about your daily routine and what’s been going on in your life lately.
Saved the timeline to /Users/piek/Desktop/Leolani/cltl-apps/cltl-custom-diabetes/notebooks/chat_logs/timeline_Jan_20261105-153137.png
[kg_chat_gui] conversation saved to /Users/piek/Desktop/Leolani/cltl-apps/cltl-custom-diabetes/notebooks/chat_logs/chat1793888903_turns_20261105-153841.json
[kg_chat_gui] statistics saved to  /Users/piek/Desktop/Leolani/cltl-apps/cltl-custom-diabetes/notebooks/chat_logs/chat1793888903_stats_20261105-153841.json
[kg_chat_gui] gap log saved to     /Users/piek/Desktop/Leolani/cltl-apps/cltl-custom-diabetes/notebooks/chat_logs/chat1793888903_gaplog_20261105-153841.json
Talked with Jan; the last conversation before this one was on 2026-11-03


Inspect what was extracted, pushed, and where each agent reply came from:

In [1]:
print(f"{len(kg_session.turns)} turns, {len(kg_session.annotations)} annotated, "
      f"{len(kg_session.kg_pushes)} pushes to the knowledge graph")
print("reply sources:", kg_session.reply_sources)
print("saturated:", tracker.is_saturated(), "| wrapped up:", tracker.wrapped_up)
print("reported vs. target, per topic:",
      {t: (tracker.reported[t], target["expected_count"]) for t, target in tracker.targets.items()})
print("catch-up questions actually asked:", tracker.asked_log)
kg_session.kg_pushes


NameError: name 'kg_session' is not defined

### Timeline after the conversation

The same timeline, redrawn after the chat window is closed, with `highlight_chat=CHAT_ID`:
everything this chat added to the KG has a **dark outline** (activities are minted under the chat
id, `chat<CHAT_ID>.<n>`). Compare it with the timeline at the start to see what the catch-up
filled in. Typically there are new entries between the *last conversation* and *now* lines, in
the lanes of the topics the tracker asked about. A lane that is still empty there is a topic that
was asked about but not answered with anything extractable (see `tracker.asked_log` above).

In [6]:
plot_patient_timeline_from_kg(brain, HUMAN, current_date=CURRENT_DATE, days=TIMELINE_DAYS,
                              last_conversation_date=last_conversation_date,
                              highlight_chat=CHAT_ID,
                              title=f"What was the case for {HUMAN}, after chat {CHAT_ID}")
plt.show()

NameError: name 'plot_patient_timeline_from_kg' is not defined

`kg_session.turn_log` has the same per-turn breakdown `kg_intent_chat.ipynb` prints live: what
each turn pushed, which intent (if any) matched, and which requirement its reply was about
(`None` for a default- or saturation-loop-driven reply -- both are indistinguishable from
`turn_log`'s own point of view, since neither comes from `intent_gap_finder`;
`kg_session.reply_sources`/`tracker.asked_log` above are what tell them apart).

In [5]:
kg_session.turn_log

[{'turn': 1,
  'speaker': 'agent',
  'utterance': 'Hi Jan, it’s been about a week since we last talked. Last time we checked in on how you were feeling physically, your meds and treatment, what you eat and drink, your exercise, symptoms, mood, blood sugar measurements, sleep, and social life. Since then, what’s changed—if anything—in what, when, or how you’ve been eating?',
  'triples_pushed': [{'subject': 'eating',
    'predicate': 'agent_patient',
    'object': 'you'}],
  'gap_queries': [],
  'selected_gap': None},
 {'turn': 2,
  'speaker': 'Jan',
  'utterance': 'How did I sleep?',
  'triples_pushed': [],
  'gap_queries': [],
  'selected_gap': None},
 {'turn': 3,
  'speaker': 'agent',
  'utterance': 'From what you’ve told me, you normally sleep about 6 hours a night. On Sunday 20 September you slept about 12 hours, which was a big change, and around then you were still feeling tired and sometimes fell asleep watching TV. Over the three days leading up to 26 September, you said you we

Save the turns, plus an **intent log** under `notebooks/intents_log/` summarizing the whole
session: the gap and the topics identified before the chat began (`topics_at_start`), how each of
those topics actually fared live (`topics_covered` -- reported/asked counts, whether its target
was met or it was capped out, and the tracker's own `asked_log`), and every
`intent_gap_finder.py` intent the per-turn flow consulted along the way (`intents_covered`) --
see `catch_up_from_kg.save_intent_log()` for the exact shape.

In [6]:
# session.turns: the turns about the name, then the catch-up chat.
save_turns(session.turns, "intents_log/kg_catchup_intent_turns.json")
catch_up.save_intent_log(kg_session, tracker, catch_up_topics, CURRENT_DATE, last_conversation_date)


Wrote 13 turns to intents_log/kg_catchup_intent_turns.json
Wrote intent log to intents_log/chat1791271636_intents_20261006-093053.json


PosixPath('intents_log/chat1791271636_intents_20261006-093053.json')